In [ ]:
import plotting  # this has to be run from /scripts/cococo

import mqt.qecc.cococo.utils_routing as utils
from mqt.qecc.cococo import hill_climber, layouts

## Sample Run for Hill Climbing 

#### Choose a layout

In [ ]:
# layout_type = "pair"
# m = 3
# n = 4
# factories =[
# (12, 7),
# (14, -1),
# (5, -1),
# (3, 7),]
# remove_edges = False
# g, data_qubit_locs, factory_ring = layouts.gen_layout_scalable(layout_type, m, n, factories, remove_edges)

# t=4

In [ ]:
# layout_type = "hex"
# m = 3
# n = 3
# factories = [
#    (4, 9),
#    (7, 10),
#    (18, 9),
#    (10, 0),
#    (-1, -2)
# ]
# g, data_qubit_locs, factory_ring = layouts.gen_layout_scalable(layout_type, m, n, factories, remove_edges=False)
#
# t=4

In [ ]:
layout_type = "hex"
m = 2
n = 2
factories = [(5, -1), (10, 0), (7, 7), (12, 5), (-2, 2)]
g, data_qubit_locs, factory_ring = layouts.gen_layout_scalable(layout_type, m, n, factories, remove_edges=False)
t = 4

In [ ]:
size = (10, 7)
plotting.plot_lattice(g, size=size, data_qubit_locs=data_qubit_locs, factory_locs=factories)

### Take the given circuit or sample another

In [ ]:
q = len(data_qubit_locs)
# pairs = circuit_construction.generate_random_circuit(q, q, tgate=True, ratio = 0.8)
# print(len(pairs))

In [ ]:
pairs = [
    (1, 5),
    0,
    (10, 3),
    (8, 18),
    (13, 2),
    (4, 20),
    (9, 6),
    (11, 23),
    15,
    7,
    (16, 21),
    (22, 23),
    (0, 10),
    (23, 19),
    (10, 12),
    (2, 19),
    (21, 1),
    (13, 4),
    22,
    (14, 13),
    (22, 1),
    (15, 11),
    (22, 20),
    (6, 11),
    (10, 8),
    13,
    19,
    (17, 20),
]

### Run the Hill Climber for the initial mapping

In [ ]:
custom_layout = [data_qubit_locs, g]

hc = hill_climber.HillClimbing(
    max_restarts=20,
    max_iterations=50,
    circuit=pairs,
    metric="exact",
    t=t,
    custom_layout=custom_layout,
    use_dag=True,
    valid_path="cc",
    possible_factory_positions=factories,
    num_factories=len(factories),
    optimize_factories=False,
)

parallel = True
processes = 8  # depending on your hardware
prefix = "./"  # adapt the path depending on where you want to have stored the output
suffix = "test_hc"
best_solution, best_score, best_rep, score_history = hc.run(prefix, suffix, parallel, processes)
print(f"Best solution: {best_solution}")
print(f"Best score: {best_score}")
print(f"To which repetition of the random restarts does the best score belong? {best_rep}")

### Run the Standard Routing with a standard layout

In [ ]:
# standard layout
layout = {}
for i, j in zip(range(len(data_qubit_locs)), data_qubit_locs, strict=False):
    layout.update({i: (int(j[0]), int(j[1]))})
terminal_pairs = layouts.translate_layout_circuit(pairs, layout)  # let's stick to the simple layout

In [ ]:
router = utils.BasicRouter(g, data_qubit_locs, factories, valid_path="cc", t=t, metric="exact", use_dag=True)
layers = router.split_layer_terminal_pairs(terminal_pairs)
vdp_layers, _ = router.find_total_vdp_layers_dyn(layers, data_qubit_locs, router.factory_times, layout, testing=True)
print("Len of schedule without mapping optimization: ", len(vdp_layers))

### Run the Standard Routing with optimized layout

In [ ]:
layout_cleaned = {key: val for key, val in best_solution.items() if key != "factory_positions"}
factories2 = best_solution["factory_positions"]

terminal_pairs = layouts.translate_layout_circuit(pairs, layout_cleaned)  # let's stick to the simple layout

In [ ]:
router = utils.BasicRouter(g, data_qubit_locs, factories2, valid_path="cc", t=t, metric="exact", use_dag=True)
layers = router.split_layer_terminal_pairs(terminal_pairs)
vdp_layers2, _ = router.find_total_vdp_layers_dyn(
    layers, data_qubit_locs, router.factory_times, layout_cleaned, testing=True
)
print("Len of schedule with mapping opt: ", len(vdp_layers2))

In [ ]:
print("Is the optimized mapping better?", len(vdp_layers) - len(vdp_layers2))

### Plot the routing

In [ ]:
cutoff = 3

for i, vdp_dict in enumerate(vdp_layers2[:cutoff]):
    print(f"=====layer = {i}====")
    if vdp_dict:
        plotting.plot_lattice_paths(
            g,
            vdp_dict,
            {},
            layout=layout_cleaned,
            factory_locs=factories2,
            size=size,
        )